# Day 5 · Semantic Search — Finding Meaning, Not Just Keywords

**Goal:** Build a working semantic search engine from scratch. Embed a 20-sentence knowledge base, retrieve the top-3 results for any query, and compare semantic search against traditional keyword matching.

**Why it matters:** Semantic search is the engine behind RAG (retrieval-augmented generation), modern recommendation systems, and most AI products that "find relevant things." Unlike keyword search, it matches *meaning* — so "affordable housing" can find "low-cost apartments" even when no words overlap.

In [ ]:
# ── 1. Install & Import ──────────────────────────────────────────────
!pip install -q sentence-transformers

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import re

print("ready ✅")

In [ ]:
# ── 2. Dataset — 20 sentences about Space Exploration ────────────────
# Topic chosen: Space Exploration & Astronomy

knowledge_base = [
    "NASA's Artemis program aims to return humans to the Moon by the mid-2020s.",                   # 0
    "SpaceX's Starship is designed to be a fully reusable rocket for deep space missions.",         # 1
    "The James Webb Space Telescope captures infrared images of distant galaxies.",                 # 2
    "Mars rovers like Perseverance search for signs of ancient microbial life.",                    # 3
    "The International Space Station orbits Earth at roughly 28,000 km/h.",                        # 4
    "Black holes warp spacetime so strongly that not even light can escape.",                       # 5
    "Astronauts experience muscle and bone loss during extended stays in microgravity.",            # 6
    "Europa, a moon of Jupiter, may harbor a subsurface ocean suitable for life.",                  # 7
    "Solar panels on spacecraft convert sunlight into electrical energy for instruments.",          # 8
    "The Voyager probes, launched in 1977, have entered interstellar space.",                       # 9
    "Exoplanets in the habitable zone could potentially support liquid water.",                     # 10
    "Rocket propulsion works by expelling mass at high velocity — Newton's third law.",             # 11
    "The cosmic microwave background is the afterglow radiation from the Big Bang.",                # 12
    "Space debris poses a growing collision risk to operational satellites.",                        # 13
    "Elon Musk envisions establishing a self-sustaining city on Mars.",                             # 14
    "Gravitational waves were first detected by LIGO in 2015.",                                    # 15
    "Astronaut training includes underwater simulations to mimic weightlessness.",                  # 16
    "The Hubble Space Telescope has been observing the universe since 1990.",                       # 17
    "Nuclear thermal propulsion could cut Mars travel time in half.",                               # 18
    "The search for extraterrestrial intelligence uses radio telescopes to scan for signals."       # 19
]

print(f"Knowledge base: {len(knowledge_base)} sentences\n")
for i, s in enumerate(knowledge_base):
    print(f"  [{i:>2}] {s}")

In [ ]:
# ── 3. Embed all sentences & store as vectors ───────────────────────
model = SentenceTransformer('all-MiniLM-L6-v2')  # same model from Day 4

# Encode the entire knowledge base into dense vectors
kb_embeddings = model.encode(knowledge_base, show_progress_bar=True)

print(f"\nVector store shape: {kb_embeddings.shape}")
print(f"  → {kb_embeddings.shape[0]} documents, each a {kb_embeddings.shape[1]}-dimensional vector")
print(f"\nSample vector (sentence 0, first 8 dims):")
print(f"  {np.round(kb_embeddings[0][:8], 4)}")

In [ ]:
# ── 4. Semantic search function ─────────────────────────────────────

def semantic_search(query: str, corpus: list[str], corpus_embeddings: np.ndarray,
                    model: SentenceTransformer, top_k: int = 3) -> list[tuple[int, float, str]]:
    """
    Embed the query, compute cosine similarity against every document
    vector, and return the top-k most similar results.

    Returns:
        List of (index, score, text) tuples, sorted by descending similarity.
    """
    # 1. Embed the query
    query_vec = model.encode([query])               # shape (1, 384)

    # 2. Cosine similarity against every stored vector
    scores = cosine_similarity(query_vec, corpus_embeddings)[0]  # shape (N,)

    # 3. Get top-k indices
    top_indices = np.argsort(scores)[::-1][:top_k]

    # 4. Return results
    results = [(int(i), float(scores[i]), corpus[i]) for i in top_indices]
    return results

print("semantic_search() defined ✅")

In [ ]:
# ── 5. Keyword search function (baseline) ──────────────────────────

def keyword_search(query: str, corpus: list[str], top_k: int = 3) -> list[tuple[int, float, str]]:
    """
    Simple keyword search: score each document by the fraction of
    query words that appear in it (case-insensitive).

    Returns:
        List of (index, score, text) tuples, sorted by descending score.
    """
    query_tokens = set(re.findall(r'\w+', query.lower()))

    scored = []
    for idx, doc in enumerate(corpus):
        doc_tokens = set(re.findall(r'\w+', doc.lower()))
        # Score = fraction of query words found in the document
        if len(query_tokens) == 0:
            score = 0.0
        else:
            score = len(query_tokens & doc_tokens) / len(query_tokens)
        scored.append((idx, score, doc))

    scored.sort(key=lambda x: x[1], reverse=True)
    return scored[:top_k]

print("keyword_search() defined ✅")

In [ ]:
# ── 6. Helper: side-by-side comparison ──────────────────────────────

def compare_search(query: str):
    """Run both search methods on the same query and print results side by side."""
    sem_results = semantic_search(query, knowledge_base, kb_embeddings, model, top_k=3)
    kw_results  = keyword_search(query, knowledge_base, top_k=3)

    print("=" * 90)
    print(f"  QUERY: \"{query}\"")
    print("=" * 90)

    print("\n🔍  SEMANTIC SEARCH (meaning-based)")
    print("-" * 60)
    for rank, (idx, score, text) in enumerate(sem_results, 1):
        print(f"  #{rank}  [doc {idx:>2}]  score={score:.4f}")
        print(f"        \"{text}\"")

    print("\n📝  KEYWORD SEARCH (word-overlap)")
    print("-" * 60)
    for rank, (idx, score, text) in enumerate(kw_results, 1):
        print(f"  #{rank}  [doc {idx:>2}]  score={score:.4f}")
        print(f"        \"{text}\"")

    print()

print("compare_search() defined ✅")

---
## Example Queries + Top-3 Outputs

We'll run **5 queries** that highlight different strengths and weaknesses of semantic vs keyword search.

In [ ]:
# ── Query 1: Direct keyword overlap ─────────────────────────────────
# Both methods should work well here — the query shares exact words.
compare_search("Mars rover searching for life")

In [ ]:
# ── Query 2: Paraphrased / synonym-heavy ────────────────────────────
# No overlapping keywords, but the MEANING matches.
# Semantic search should shine; keyword search will struggle.
compare_search("reusable spacecraft for interplanetary travel")

In [ ]:
# ── Query 3: Abstract / conceptual ──────────────────────────────────
# The query asks about a *concept* — health risks of living in space.
# Keyword search won't match; semantic search understands the idea.
compare_search("health risks of living in space")

In [ ]:
# ── Query 4: Question-style natural language ────────────────────────
# Users naturally ask questions — semantic models handle this well.
compare_search("Is there liquid water on any moon in our solar system?")

In [ ]:
# ── Query 5: Looking for alien signals ──────────────────────────────
# Tests whether each method can find SETI-related content.
compare_search("listening for alien radio transmissions")

---
## Analysis: Semantic Search vs Keyword Search

| Aspect | Keyword Search | Semantic Search |
|---|---|---|
| **How it works** | Counts exact word overlaps between query and document | Encodes query and documents into dense vectors, ranks by cosine similarity |
| **Handles synonyms?** | ❌ No — "spacecraft" ≠ "rocket" | ✅ Yes — understands they mean similar things |
| **Handles paraphrasing?** | ❌ No — completely different words = zero score | ✅ Yes — captures the underlying intent |
| **Handles questions?** | ⚠️ Poorly — question words add noise | ✅ Naturally — transformer models are trained on Q&A pairs |
| **Speed** | ⚡ Very fast (string matching) | 🐢 Slower (requires model inference + similarity computation) |
| **Requires ML model?** | No | Yes (pre-trained transformer) |
| **Best for** | Exact-match lookups, filters | Understanding user intent, RAG, recommendations |

### Key Takeaways

1. **Semantic search excels when the user's words don't match the document's words** — it understands that "reusable spacecraft for interplanetary travel" is about Starship even though none of those words appear in the sentence.

2. **Keyword search fails silently** — it returns results ranked by accidental word overlap, which can be misleading (e.g., matching "space" in unrelated documents).

3. **In production, combine both** — many real systems use keyword search for exact-match filtering (BM25) plus semantic search for re-ranking. This is called **hybrid search** and gives you the best of both worlds.

4. **This is the foundation of RAG** — in retrieval-augmented generation, the semantic search step retrieves relevant context, which is then fed to an LLM to generate an answer. The quality of retrieval directly determines the quality of the generated answer.

---
## Reflection

**What clicked today:**
- Semantic search is literally just "embed everything, then find the nearest vectors to your query." The magic is entirely inside the embedding model — it maps meaning into geometry.
- Keyword search is fundamentally limited because language is full of synonyms, paraphrases, and implied meaning. Two sentences can mean the same thing with zero word overlap.

**What surprised me:**
- How well the model handles *questions* as queries — it correctly maps "Is there liquid water on any moon?" close to the Europa sentence about subsurface oceans, despite very different surface forms.
- Keyword search can be *actively misleading* — it returns results that share words but not meaning, giving false confidence.

**Connection to Day 4:**
- Day 4 showed that embeddings capture meaning (similar sentences = similar vectors). Day 5 turns that insight into a *useful tool* — a search engine. This is exactly how RAG pipelines work: embed your knowledge base, embed the user's question, retrieve the closest matches, then pass them to an LLM.